In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import get_points_array, project_line

In [ ]:
import sys
sys.path.append('./original_code')
from original_code.plot import plot_results, plot_nonhermitian_bands, plot_result_by_band

In [ ]:
points = get_points_array('./data/arrays/0.npz')

In [ ]:
plot_results(k_vals=points[:, 0], omega_vals=points[:, 1])

# Trying clustering
---

Not currently being worked on. Try this for the non-Hermitian case, maybe.

# Trying to count intersections of lines
---

In [ ]:
point_clusters = project_line(points=points, x_coord=-np.pi / 4, n_steps=1_000)

In [ ]:
len(point_clusters)

In [ ]:
# Plot detected points

cmap = plt.get_cmap('viridis')

plt.figure(figsize=(8, 6))
plt.scatter(points[:,0], points[:,1], s=1, c='k', alpha=0.5)

# Plot found clusters
for i in range(len(point_clusters)):
    color = cmap(i / len(point_clusters))
    plt.scatter(point_clusters[i][:,0], point_clusters[i][:,1], s=20, color=color, label=f"Cluster {i}", alpha=1.0)


plt.xlabel('k (Bloch Wavenumber)', fontsize=14)
plt.ylabel('ω (Frequency)', fontsize=14)
plt.title('Photonic Crystal Band Structure (Non-Hermitian Case)', fontsize=14)
plt.xlim(-np.pi, np.pi)
plt.ylim(0, 0.6)
plt.xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
            [r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])
plt.grid(True, linestyle='--', alpha=0.6)
plt.tick_params(axis='both', labelsize=14)
plt.show()

KD Tree Method (Not current work)
---

In [ ]:
from scipy.spatial import KDTree

In [ ]:
def build_band(points, start_point, num=10_000, seen=None):
    start_idx = np.where(np.all(points == start_point, axis=1))[0].item()
    
    if not seen:
        seen = {start_idx}
    
    band = np.array([start_point])
    tree = KDTree(points)
    
    for i in range(num):
        _, indices = tree.query(band[-1], k=20)
        
        all_seen = True
        for index in indices:
            if index not in seen:
                if points[index][0] < band[-1][0]:
                    continue
                all_seen = False
                seen.add(index)
                band = np.vstack((band, points[index]))
        
        if all_seen:
            return band, seen
    
    return band, seen

In [ ]:
seen_y_vals = []
bands = []

for i in range(30):
    if i == 0:
        band, seen = build_band(points, start_point=points[i], num=100_000)
        bands.append(band)
        seen_y_vals.append(points[i][1])
        continue
    
    if np.any(np.abs(seen_y_vals - points[i][1]) < 1e-4):
        continue
    band, seen = build_band(points, start_point=points[i], num=100_000, seen=seen)
    bands.append(band)
    seen_y_vals.append(points[i][1])

In [ ]:
plot_result_by_band([*bands])

In [ ]:
for i, idx in enumerate(seen):
    if idx != i:
        print(f"Mismatch at  {i}")
        break

In [ ]:
print(len(seen))

In [ ]:
total = 0
for band in bands:
    total += band.shape[0]
print(total)

In [ ]:
unique_rows, counts = np.unique(points, axis=0, return_counts=True)

---